# Chapter 6 of Book Hands on LLM

In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, pipeline

In [ ]:
model = AutoModelForCausalLM.from_pretrained(
    "microsoft/Phi-3-mini-4k-instruct",
    device_map="cuda",
    torch_dtype="auto",
    trust_remote_code=True,
)

In [ ]:
tokenizer = AutoTokenizer.from_pretrained("microsoft/Phi-3-mini-4k-instruct")

In [ ]:
pipe = pipeline(
    "text-generation",
    model=model,
    tokenizer=tokenizer,
    return_full_text=False,
    max_new_tokens=500,
    do_sample=False, # needs to be true if using top_p or temperature
)

In [ ]:
# Simple Prompt
messages = [
    {"role": "user", "content": "Create a funny joke about chickens."}
]

# Generate the output
output = pipe(messages)
print(output[0]["generated_text"])

In [ ]:
# how the list of dicts is transformed into a prompt template
# through the use of the method apply_chat_template
promt = pipe.tokenizer.apply_chat_template(messages, tokenize=False)
print(promt)

In [ ]:
'''
using Temperature to create more diverse output; Remember,
a higher temperature increases the likelihood that less
probable tokens are generated and vice versa
'''
# low temperature
output = pipe(messages, do_sample=True, temperature=0.1)
print(output[0]["generated_text"])
# high temperature
output = pipe(messages, do_sample=True, temperature=1)
print(output[0]["generated_text"])

In [ ]:
'''
using Top_p (aka nucleus sampling) to create more diverse output; Top_p
is a sampling technique that controls which subset of tokens (the nucleus) the
LLM can consider. It will consider tokens until it reaches their cumulative
probability; So if top_p is set to 0.1 the llm will consider tokens until it
reaches that value, if set to 1 it will consider all tokens
'''
# low top_p
output = pipe(messages, do_sample=True, top_p=0.1)
print(output[0]["generated_text"])
# high top_p
output = pipe(messages, do_sample=True, top_p=1)
print(output[0]["generated_text"])



---


# Creating Complex Prompt through Iteration

A **general prompt** can include the followng components: 1) an instruction; 2) data; and 3)output indicators.

A **complex prompt** can include the following components: 1) persona; 2) instruction; 3) context; 4) format; 5) audience; 6) tone; and 7) data


*   **Persona**: Describe what role the LLM should take on.
*   **Instruction**: The task itself. Make sure this is as specific as possible. We do not want to leave much room for interpretation.
*   **Context**: Additional information describing the context of the task
*   **Format**: The format the LLM should use to output the generated text
*   **Audience**: The target of the generated text. This also describes the level of the generated output.
*   **Tone**: The tone of voice the LLM should use in the generated text.
*   **Data**: The main data related to the task itself


---




In [ ]:
# Creating a Complex prompt from Prompt Components
persona = "You are an expert in Large Language models. You excel at breaking down complex papers into digestible summaries.\n"
instruction = "Summarize the key findings of the paper provided.\n"
context = "Your summary should extract the most crucial points that can help researchers quickly understand the most vital information of the paper.\n"
data_format = "Create a bullet-point summary that outlines the method. Follow this up with a concise paragraph that encapsulates the main results.\n"
audience = "The summary is designed for busy researchers that quickly need to grasp the idea\n"
tone = "The tone should be professional and clear.\n"
text = """Pamir was a four-masted barque built for the German shipping
          company F. Laeisz. One of their famous Flying P-Liners,
          she was the last commercial sailing ship to round Cape Horn,
          in 1949. By 1957, she had been outmoded by modern bulk carriers and could not operate at a profit.
          Her shipping consortium's inability to finance much-needed repairs or to recruit sufficient sail-trained officers caused severe technical difficulties.
          On 21 September 1957, she was caught in Hurricane Carrie and sank off the Azores, with only six survivors rescued after an extensive search"""
data = f"Text to summarize: {text}"

In [ ]:
# Iteration one of prompt components
query = instruction + context + data
output = pipe(query, do_sample=False)
print(output[0]['generated_text'])

In [ ]:
# Iteration two of prompt components
query = persona+instruction + context + data
output = pipe(query, do_sample=False)
print(output[0]['generated_text'])

In [ ]:
# Iteration two of prompt components
query = persona+instruction + context + data + data_format
output = pipe(query, do_sample=False)
print(output[0]['generated_text'])

# In-Context Learning: Providing Examples

For in-context learning instead of describing the task we show it the task by
giving it correct examples. The number of examples determines the type of in context prompting

*   **One-shot prompt**: prompting with a single example
*   **Few-shot prompt**: prompting with more than one example

This method was invented in the paper [Tom Brown et al. “Language models are few-shot learners.” Advances in Neural Information Processing Systems 33 (2020): 1877–1901.](https://arxiv.org/pdf/2005.14165)

To do so in the prompt we have to differentiate between our question in our list of dicts with the key **user** and the answer we want the model to produce with the key **assistant**

---



In [ ]:
'''
In this example the goal of the prompt is to generate a
sentence with a made-up word using one-shot prompting
'''
# A single example of using a made-up word in a sentence
one_shot_prompt = [
    {
        "role": "user",
        "content": "A 'Gigamuru' is a type of Japanese musical instrument. An example of a sentence that uses the word Gigamuru is:"
    },
    {
        "role": "assistant",
        "content": "I have a Gigamuru that my uncle gave me as a gift. I love to play it at home."
    },
    {
        "role": "user",
        "content": "To 'screeg' something is to swing a sword at it. An example of a sentence that uses the word screeg is:"
    }
]
# seeing what the above one shot prompt looks like as a template
print(tokenizer.apply_chat_template(one_shot_prompt, tokenize=False))

In [ ]:
outputs = pipe(one_shot_prompt)
print(outputs[0]["generated_text"])

# Chain Prompting: Breaking up the Problem

Chain prompting is a technique that takes the **output** of one prompt and uses it as **input** into the next prompt. This is done by calling  an LLM multiple times.

Good for the following:

*   **Parallel prompts**:Create multiple prompts in parallel and do a final pass to merge them.
*   Generating text based on previous generated text e.g., sections within a brief



---



In [ ]:
'''
In this example we want to generate a businees name, slogan and then
sales pitch using chain prompting
'''
# prompt to create name for business
name_prompt = [
    {"role": "user", "content": "Create a name for a law firm that leverages LLMs and AI to always win for its clients."}
]
name = pipe(name_prompt, do_sample=True, top_p=0.5)
print(name[0]["generated_text"])

# prompt to create slogan for business
slogan_prompt = [
    {"role": "user", "content": f"Create a slogan for the following law firm '{name}'"}
]
slogan = pipe(slogan_prompt)
print(slogan[0]["generated_text"])

# prompt to create sales pitch for business
sales_prompt = [
    {"role": "user", "content": f"Generate a very short sales pitch for the following slogan: '{slogan}'"}
]
output = pipe(sales_prompt)
print(output[0]["generated_text"])

# Chain-of-Thought: Think Before Answering

This technique uses LLMs to create "reasoning capabiliities" in that it mimics the reasoning process of system 2 thinking. **System 2 thinking is a concious, slow, and logical process, akin to brainstorming and self-reflection**. Previously, the techniques used in this notebook were using system 1 thinking in which the llm was answering in an automatic manner without self-reflection.   

The method of chain-of-thought prompting aims to have the LLM **think first** before answering the question. This method was invented in the paper: [Jason Wei et al. “Chain-of-thought prompting elicits reasoning in large language models.” Advances in Neural Information Processing Systems 35 (2022): 24824–24837.](https://proceedings.neurips.cc/paper_files/paper/2022/file/9d5609613524ecf4f15af0f7b31abca4-Paper-Conference.pdf)

As the authors detail:

> We explore how generating a chain of thought—**a series of intermediate reasoning steps—significantly improves the ability of large language models to perform complex reasoning**...experiments on three large language models show that chain-of-thought prompting improves performance on a range of arithmetic, commonsense, and symbolic reasoning tasks.


> large language models offer the exciting prospect of in-context few-shot learning via prompting. **That is, instead of finetuning a separate language model checkpoint for each new task, one can simply “prompt” the model with a few
input-output exemplars demonstrating the task**.

Chain-of-thought prompting has several attractive properties as an approach for facilitating reasoning
in language models.

*  First, chain of thought, in principle, allows models to decompose multi-step problems into intermediate steps, which means that additional computation can be allocated to problems that require more reasoning steps
*  Second, a chain of thought provides **an interpretable window into the behavior of the model,suggesting how it might have arrived at a particular answer** and providing opportunities to debug where the reasoning path went wrong (**although fully characterizing a model’s computations that support an answer remains an open question**).
* Third, chain-of-thought reasoning  is potentially applicable (at least
in principle) **to any task that humans can solve via language**.






---



In [ ]:
# Answering with chain-of-thought
cot_prompt = [
    {"role": "user", "content": "Roger has 5 tennis balls. He buys 2 more cans of tennis balls. Each can has 3 tennis balls. How many tennis balls does he have now?"},
    {"role": "assistant", "content": "Roger started with 5 balls. 2 cans of 3 tennis balls each is 6 tennis balls. 5 + 6 = 11. The answer is 11."}, # the reasoning component
    {"role": "user", "content": "The cafeteria had 23 apples. If they used 20 to make lunch and bought 6 more, how many apples do they have?"}
]

# Generate the output
outputs = pipe(cot_prompt)
print(outputs[0]["generated_text"])

# Testing Phi-3 for Generating Boolean Queries

LLMs are unique in that they have great generalization performance for a range of tasks and interestingly, this can be done through effective prompt engineering. Case in point is the new research area of using LLMs to construct search queires for information retrival purposes i.e. finding the most relevant documents based on a boolean based search query. This is a very complex task that is an open research question that has not been solved. The leading paper that actually showed that this type of task could be done (using pre-trained LLMs) was   

*   [Can ChatGPT Write a Good Boolean Query for Systematic Review Literature Search?](https://arxiv.org/pdf/2302.03495)

As the authors of the paper detail:

> Through a number of extensive experiments on standard test collections for the task, we find that
ChatGPT is capable of generating queries that lead to high search precision, although trading-off this for recall. Overall, our
study demonstrates the potential of ChatGPT in generating effective Boolean queries for systematic review literature search.

Four different methods were used to engineer the textual prompts that were fed into the ChatGPT LLM model (which the authors of [A Reproducibility and Generalizability Study of Large Language
Models for Query Generation](https://arxiv.org/pdf/2411.14914) determined to be ChatGPT 3.5), namely:

1.   **zero-shot prompts**: The tested prompts were of the following form in which {review_title} was substitued with the title of the systematic review:
  *   (q1): For a systematic review titled {review_title}, can you generate a  systematic review Boolean query to find all included studies on PubMed for the review topic?
  *   (q2): You are an information specialist who develops Boolean queries for systematic reviews. You have extensive
experience developing highly effective queries for searching the medical literature. Your specialty is
developing queries that retrieve as few irrelevant documents as possible and retrieve all relevant documents
for your information need. Now you have your information need to conduct research on {review_title}.
Please construct a highly effective systematic review Boolean query that can best serve your information
need.
  *  (q3): Imagine you are an expert systematic review information specialist; now you are given a systematic review
research topic, with the topic title “{review_title}”. Your task is to generate a highly effective systematic
review Boolean query to search on PubMed (refer to the professionally made ones); the query needs to be
as inclusive as possible so that it can retrieve all the relevant studies that can be included in the research
topic; on the other hand, the query needs to retrieve fewer irrelevant studies so that researchers can spend
less time judging the retrieved documents.

2.   **one-shot prompts**: The tested prompts were of the following form in which {review_title} was substitued with the title of the systematic review,{example_review_title} was substituted with the systematic review topic, and {example_review_query} was substituted with a sample Boolean query:
  *   (q4): You are an information specialist who develops Boolean queries for systematic reviews. You have extensive
experience developing highly effective queries for searching the medical literature. Your specialty is
developing queries that retrieve as few irrelevant documents as possible and retrieve all relevant documents
for your information need. You are able to take an information need such as: “{example_review_title}”
and generate valid pubmed queries such as: “{example_review_query}". Now you have your information
need to conduct research on “{review_title}”, please generate a highly effective systematic review Boolean
query for the information need.
  *   (q5): You are an information specialist who develops Boolean queries for systematic reviews. You have extensive experience developing highly effective queries for searching the medical literature. Your specialty
is developing queries that retrieve as few irrelevant documents as possible and retrieve all relevant
documents for your information need. A professional information specialist will extract PICO elements
from information needs in a common practice in constructing a systematic review Boolean query. PICO
means Patient/ Problem, Intervention, Comparison and Outcome. PICO is a format for developing a
good clinical research question prior to starting one’s research. It is a mnemonic used to describe the
four elements of a sound clinical foreground question. You are able to take an information need such as:
“{example_review_title}" and you generate valid pubmed queries such as: “{example_review_query}". Now
you have your information need to conduct research on “{review_title}”. First, extract PICO elements from
the information needs and construct a highly effective systematic review Boolean query that can best
serve your information need.

3.  **query refinement**: The tested prompts were of the following form in which {initial_query} was the initial Boolean query we want ChatGPT to refine:
  *   (q6): For a systematic review seed Boolean query: "{initial_query}", This query retrieves too many irrelevant
documents and too few relevant documents about the information need: “{review_title}”, Please correct
this query so that it can retrieve fewer irrelevant documents and more relevant documents.
  *   (q7): For a systematic review seed Boolean query: “{example_review_initial_query}" ,This query retrieves
too many irrelevant documents and too few relevant documents about the information need: “{example_review_title}”, therefore it should be corrected to: “{example_review_refined_query}”. Now your task is
to correct a systematic review Boolean query: "{initial_query}" for information need “{review_title}”, so it
can retrieve fewer irrelevant documents and more relevant documents.

4. **Guided Prompts**:
  *   See Table 3 of [Can ChatGPT Write a Good Boolean Query for Systematic Review Literature Search?](https://arxiv.org/pdf/2302.03495) for the detailed steps


Interestingly, the authors of [Can ChatGPT Write a Good Boolean Query for Systematic Review Literature Search?](https://arxiv.org/pdf/2302.03495) found that combining (q4) and (q7-objective) produced the highest precision and F-measure noting the following:

> Specifically, the use of ChatGPT for query refinement
leads to an increase in precision and F-measure, while obtaining a lower recall. Therefore, it is crucial to first
create a seed query with a high recall, and then use ChatGPT to refine the query in order to achieve highly
effective Boolean queries.

Instead of doing a reproducibility experiment as detailed in [A Reproducibility and Generalizability Study of Large Language
Models for Query Generation](https://arxiv.org/pdf/2411.14914) with ChatGPT I decided to try out generating boolean queries using the [Phi-3 mini model](https://huggingface.co/microsoft/Phi-3-mini-4k-instruct) which was developed by [Microsoft](https://arxiv.org/pdf/2404.14219). As the model card details:

> The Phi-3-Mini-4K-Instruct is a 3.8B parameters, lightweight, state-of-the-art open model trained with the **Phi-3 datasets that includes both synthetic data and the filtered publicly available websites data** with a focus on high-quality and reasoning dense properties. The model belongs to the Phi-3 family with the Mini version of 4K being the context length (in tokens) that it can support.

The phi-3-mini model is a transformer decoder architecture, built upon a similar block structure as Llama-2 and uses the same tokenizer with a vocabulary size of 32,064. The model inputs a 3072 embedding vector and has 32 attention heads and 32 layers and was trained using bfloat16 precision. The architecture is detailed below:

```
Phi3ForCausalLM(
  (model): Phi3Model(
    (embed_tokens): Embedding(32064, 3072, padding_idx=32000)
    (embed_dropout): Dropout(p=0.0, inplace=False)
    (layers): ModuleList(
      (0-31): 32 x Phi3DecoderLayer(
        (self_attn): Phi3Attention(
          (o_proj): Linear(in_features=3072, out_features=3072, bias=False)
          (qkv_proj): Linear(in_features=3072, out_features=9216, bias=False)
          (rotary_emb): Phi3RotaryEmbedding()
        )
        (mlp): Phi3MLP(
          (gate_up_proj): Linear(in_features=3072, out_features=16384, bias=False)
          (down_proj): Linear(in_features=8192, out_features=3072, bias=False)
          (activation_fn): SiLU()
        )
        (input_layernorm): Phi3RMSNorm()
        (resid_attn_dropout): Dropout(p=0.0, inplace=False)
        (resid_mlp_dropout): Dropout(p=0.0, inplace=False)
        (post_attention_layernorm): Phi3RMSNorm()
      )
    )
    (norm): Phi3RMSNorm()
  )
  (lm_head): Linear(in_features=3072, out_features=32064, bias=False)
)


```

Instead of creating boolean queries for medical based retrievals I modified the prompts to create boolean queries for general searching on google for articles based on the [pamir](https://en.wikipedia.org/wiki/Pamir_(ship) a topic and a ship that has some significance in my family




In [ ]:
# zero-shot prompt using the q1 format
review_title = "Pamir was a four-masted barque built for the German shipping company F. Laeisz."
q1 = f"""For a review titled {review_title},  can you generate a Boolean query to find all articles
on google?"""
output = pipe(q1, do_sample=False)
print(output[0]['generated_text'])

In [ ]:
# zero-shot prompt using the q2 format
review_title = "Pamir was a four-masted barque built for the German shipping company F. Laeisz."
q2 = f"""
You are an information specialist who develops Boolean queries for articles.
You have extensive experience developing highly effective queries for searching ship literature.
Your specialty is developing queries that retrieve as few irrelevant articles as possible and retrieve
all relevant articles for your information need. Now you have your information need to conduct research on {review_title}.
Please construct a highly effective Boolean query that can best serve your information need.
"""
output = pipe(q2, do_sample=False)
print(output[0]['generated_text'])

In [ ]:
# zero-shot prompt using the q3 format
review_title = "Pamir was a four-masted barque built for the German shipping company F. Laeisz."
q3 = f"""
Imagine you are an expert search information specialist;
now you are given a search topic, with the topic
title “{review_title}”. Your task is to generate a highly effective Boolean query
to search on google (refer to the professionally made ones);
the query needs to be as inclusive as possible so that it can retrieve all the
relevant articles that can be included in the topic; on the other hand,
the query needs to retrieve fewer irrelevant articles so that researchers can
spend less time judging the retrieved articles.
"""
output = pipe(q3, do_sample=False)
print(output[0]['generated_text'])

In [ ]:
# one-shot prompts using the q4 format
review_title = "Pamir was a four-masted barque built for the German shipping company F. Laeisz."
example_review_title = "barque shipping age of exploration in the absence of the royal navy"
example_review_query = "((barque AND shippping) AND (review OR document) NOT royal navy)"
q4 = f"""You are an information specialist who develops Boolean queries for
searching. You have extensive experience developing highly effective
queries for searching barque ships. Your specialty is developing queries that
retrieve as few irrelevant documents as possible and retrieve all relevant documents for your information need.
You are able to take an information need such as: “{example_review_title}” and generate
valid google queries such as: “{example_review_query}". Now you have your
information need to conduct search on “{review_title}”, please generate a
highly effective Boolean query for the information need.
"""
output = pipe(q4, do_sample=False)
print(output[0]['generated_text'])

In [ ]:
# query refinement using q6
review_title = "Pamir was a four-masted barque built for the German shipping company F. Laeisz."
initial_query = "((barque AND shippping) AND (review OR document) NOT royal navy)"
q6 = f"""For a google Boolean query: "{initial_query}",
This query retrieves too many irrelevant documents and too few relevant
documents about the information need: “{review_title}”, Please correct
this google Boolean query so that it can retrieve more relevant documents.
"""
output = pipe(q6, do_sample=False)
print(output[0]['generated_text'])

In [ ]:
# query refinement using q7
example_review_title = "Pamir was a four-masted barque built for the German shipping company F. Laeisz."
example_review_initial_query = "((barque AND shippping) AND (review OR document) NOT royal navy)"
example_review_refined_query = "((barque AND shippping) AND (review OR document) NOT royal navy AND Pamir AND four-masted)"
initial_query = "(banking OR banker) AND (economic OR money) NOT crash"
review_title = "Inflated: Money, Debt and the American Dream"
q7 = f"""For a google Boolean query: “{example_review_initial_query}" ,
This query retrieves too many irrelevant documents and too few
relevant documents about the information need: “{example_review_title}”,
therefore it should be corrected to: “{example_review_refined_query}”. Now
your task is to correct a google Boolean query: "{initial_query}" for
information need “{review_title}”, so it can retrieve fewer irrelevant
documents and more relevant documents.
"""
output = pipe(q7, do_sample=False)
print(output[0]['generated_text'])